In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

def crawl_saramin():
    base_url = "https://www.saramin.co.kr/zf_user/search"
    params = {
        'search_area': 'main',
        'search_done': 'y', 
        'search_optional_item': 'n',
        'searchType': 'search',
        'searchword': '데이터분석'
    }
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    response = requests.get(base_url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    jobs = []
    job_items = soup.find_all('div', class_='item_recruit')
    
    for item in job_items:

        company_elem = item.find('strong', class_='corp_name')
        company = company_elem.get_text(strip=True) if company_elem else 'N/A'

        title_elem = item.find('h2', class_='job_tit')
        title = 'N/A'
        url = 'N/A'
        
        if title_elem:
            title_link = title_elem.find('a')
            if title_link:
                title = title_link.get_text(strip=True)
                href = title_link.get('href', '')
                if href.startswith('/zf_user'):
                    url = f"https://www.saramin.co.kr{href}"
                elif href.startswith('http'):
                    url = href
        
        condition_elem = item.find('div', class_='job_condition')
        conditions = []
        if condition_elem:
            condition_spans = condition_elem.find_all('span')
            for span in condition_spans:
                text = span.get_text(strip=True)
                if text and text not in ['↑', '↓', '|']:
                    conditions.append(text)
        
        requirement = ' | '.join(conditions) if conditions else 'N/A'
        
        jobs.append({
            'Site': 'Saramin',
            'Col_Company': company,
            'Col_Recruit': title,
            'Col_detail': requirement,
            'Col_URL': url
            
        })
    
    return pd.DataFrame(jobs)

df_saramin = crawl_saramin()

if not os.path.exists('data_tmp'):
    os.makedirs('data_tmp')

df_saramin.to_csv('data_tmp/data_saramin.csv', index=False, encoding='utf-8-sig')
